In [ ]:
import os
from pathlib import Path
# imports
import logging
import sys

import nibabel as nib
import numpy as np
import pandas as pd
import torch
from torch.utils.tensorboard import SummaryWriter

import monai
from monai.apps import download_and_extract
from monai.config import print_config
from monai.data import DataLoader, ImageDataset
from monai.transforms import (
    EnsureChannelFirst,
    Compose,
    RandRotate90,
    Resize,
    ScaleIntensity,
    Lambda
)

pin_memory = torch.cuda.is_available()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
print_config()

# Configure local data and run paths with environment variables.
DATA_ROOT = Path(os.environ.get("MRI_DATA_ROOT", "data"))
METADATA_CSV = Path(
    os.environ.get("MRI_METADATA_CSV", DATA_ROOT / "filtered_df_for_share.csv")
)
MODEL_DIR = Path(os.environ.get("MRI_MODEL_DIR", "."))
OUTPUT_DIR = Path(os.environ.get("MRI_OUTPUT_DIR", "outputs"))
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# function from chatgpt to make a 3d nifti from a 4d nifti
def extract_3d_from_4d(in_path, out_path, index=0):
    """
    Extract a 3D volume from a 4D NIfTI file along the first dimension (index).
    
    Parameters
    ----------
    in_path : str
        Path to the input 4D NIfTI (.nii or .nii.gz).
    out_path : str
        Path where the 3D NIfTI should be saved.
    index : int, optional
        Which volume to extract along the 4th dimension (default = 0).
    """
    # Load the 4D NIfTI
    img = nib.load(in_path)
    data = img.get_fdata()

    if data.ndim != 4:
        raise ValueError(f"Input file is not 4D — it has {data.ndim} dimensions.")

    if not (0 <= index < data.shape[3]):
        raise ValueError(f"Index {index} is out of range for dimension size {data.shape[3]}.")

    # Extract the 3D volume
    vol3d = data[..., index]

    # Preserve affine and header
    affine = img.affine
    header = img.header.copy()

    # Create new 3D NIfTI
    new_img = nib.Nifti1Image(vol3d, affine, header)

    # Save
    nib.save(new_img, out_path)

    return out_path

In [ ]:
# download images from flywheel
# only use this cell if you need to re-download files

# flywheel api
import flywheel
def uwhealthaz_client():
    return flywheel.Client('flywheelaz.uwhealth.org:djEfiALy55nFsQ99M4E6ZQAW8F53fYucPUqU8HQhlZ4dBfsWZ-BYOcpCQ',
                           request_timeout=1000)

fw = uwhealthaz_client()
root_dir = str(DATA_ROOT)

for index, row in flywheel_info.iterrows():
    # if index > 99:
    #     break
    # only need the session if you need anatomy labels!
    # sessin_id = 
    # session = fw.get(row['flywheel_info.session_id'])
    # session_info = session.info
    acq_id = row['flywheel_info.acq_id']
    nifti_name = row["flywheel_info.nifti_name"]
    acq = fw.get(acq_id)
    img_path = f'{root_dir}/{nifti_name}'
    acq.download_file(row['flywheel_info.nifti_name'], img_path)


In [ ]:
# get region labels from csv
flywheel_info = pd.read_csv(METADATA_CSV, low_memory=False)

root_dir = str(DATA_ROOT)

# add root dir to image file names
flywheel_info['flywheel_info.nifti_name'] = root_dir + '/' + flywheel_info['flywheel_info.nifti_name']
image_files = flywheel_info['flywheel_info.nifti_name']

# map image_labels to numerical, generate the map via pandas!
mapper = {val: i for i, val in enumerate(flywheel_info["classification.fat_suppression_label"].unique())}
flywheel_info['classification.fat_suppression_label'] = flywheel_info['classification.fat_suppression_label'].map(mapper)
image_labels = flywheel_info['classification.fat_suppression_label']

# indices to use when creating the train/val split loaders
label_groups = {label: group.index for label, group in flywheel_info.groupby('split.region')}

print(f"train: {len(image_files[label_groups['train']])}, val: {len(image_files[label_groups['val']])}, test:{len(image_files[label_groups['test']])}, total:{len(image_files)}")
print(f"category mapper: {mapper}")


In [ ]:
# Define transforms
def squeeze_to_one_channel(x):
    # if C==1, do nothing
    if x.shape[0] == 1:
        return x
    # if C>1 (e.g., DWI), keep channel 0
    return x[0:1, ...]  

train_transforms = Compose([ScaleIntensity(), 
                            EnsureChannelFirst(),
                            Lambda(squeeze_to_one_channel),
                            Resize((96, 96, 96)), 
                            RandRotate90()])

val_transforms = Compose([ScaleIntensity(), 
                          EnsureChannelFirst(), 
                          Lambda(squeeze_to_one_channel),
                          Resize((96, 96, 96))])

# Define nifti dataset, data loader
check_ds = ImageDataset(image_files=image_files.tolist(), labels=image_labels.tolist(), transform=train_transforms)
check_loader = DataLoader(check_ds, batch_size=4, num_workers=8, pin_memory=pin_memory)

im, label = monai.utils.misc.first(check_loader)
print(type(im), im.shape, label, len(label))

# create a training data loader
train_ds = ImageDataset(image_files=image_files[label_groups['train']].tolist(), labels=image_labels[label_groups['train']].tolist(), transform=train_transforms)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=16, pin_memory=pin_memory)

# create a validation data loader
val_ds = ImageDataset(image_files=image_files[label_groups['val']].tolist(), labels=image_labels[label_groups['val']].tolist(), transform=val_transforms)
val_loader = DataLoader(val_ds, batch_size=4, num_workers=8, pin_memory=pin_memory)

# create a test data loader
test_ds = ImageDataset(image_files=image_files[label_groups['test']].tolist(), labels=image_labels[label_groups['test']].tolist(), transform=val_transforms)
test_loader = DataLoader(test_ds, batch_size=2, num_workers=8, pin_memory=pin_memory)

In [ ]:
# training loop
import time

# Create DenseNet121, CrossEntropyLoss and Adam optimizer
model = monai.networks.nets.DenseNet121(spatial_dims=3, in_channels=1, out_channels=2).to(device)

loss_function = torch.nn.CrossEntropyLoss()
# loss_function = torch.nn.BCEWithLogitsLoss()  # also works with this data
optimizer = torch.optim.Adam(model.parameters(), 1e-4)

# start a typical PyTorch training
val_interval = 2
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []
writer = SummaryWriter()
max_epochs = 25

for epoch in range(max_epochs):
    t1 = time.time()
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0

    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data[0].to(device), batch_data[1].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_len = len(train_ds) // train_loader.batch_size
        # print(f"{step}/{epoch_len}, train_loss: {loss.item():.4f}")
        writer.add_scalar("train_loss", loss.item(), epoch_len * epoch + step)

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    if (epoch + 1) % val_interval == 0:
        model.eval()

        num_correct = 0.0
        metric_count = 0
        for val_data in val_loader:
            val_images, val_labels = val_data[0].to(device), val_data[1].to(device)
            with torch.no_grad():
                val_outputs = model(val_images)
                # value = torch.eq(val_outputs.argmax(dim=1), val_labels.argmax(dim=1))
                # 2025-12 csk add Dvij suggestion
                pred = val_outputs.argmax(dim=1)
                value = pred == val_labels
                metric_count += len(value)
                num_correct += value.sum().item()

        metric = num_correct / metric_count
        metric_values.append(metric)

        if metric > best_metric:
            best_metric = metric
            best_metric_epoch = epoch + 1
            torch.save(model.state_dict(), MODEL_DIR / "best_metric_model_classification3d_array.pth")
            print("saved new best metric model")

        print(f"Current epoch: {epoch+1} current accuracy: {metric:.4f} ")
        print(f"Best accuracy: {best_metric:.4f} at epoch {best_metric_epoch}")
        writer.add_scalar("val_accuracy", metric, epoch + 1)
    t2 = time.time()
    print(f'epoch time: {t2-t1}')
print(f"Training completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")
writer.close()

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)


def compute_multiclass_metrics(y_true, y_pred_prob, threshold=0.5): # Xue said she thresholded at 0.5
    """
    Computes accuracy, precision, recall, f1, roc-auc, pr-auc
    for both macro and weighted averages.
    
    Works for:
        - Binary classification (probabilities of shape [N])
        - Multiclass classification (probabilities of shape [N, C])
    """

    if y_pred_prob.ndim == 1:
        # Only 1 label so multiclass (eg fat suppression, contrast)
        num_classes = 2
        
        # Class predictions using threshold
        y_pred = (y_pred_prob >= threshold).astype(int)
        
        # Convert to 2-column probability array for multiclass-compatible metrics
        y_pred_prob_mc = np.vstack([1 - y_pred_prob, y_pred_prob]).T
    
    else:
        # Many labels so multiclass classification 
        num_classes = y_pred_prob.shape[1]
        
        # Use argmax for predicted labels (assuming softmax in your code)
        y_pred = np.argmax(y_pred_prob, axis=1)
        
        # Already correct shape
        y_pred_prob_mc = y_pred_prob
    
    metrics = {}

    # Accuracy
    metrics["accuracy"] = accuracy_score(y_true, y_pred)

    # Precision
    metrics["precision_macro"] = precision_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    metrics["precision_weighted"] = precision_score(
        y_true, y_pred, average="weighted", zero_division=0
    )

    # Recall
    metrics["recall_macro"] = recall_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    metrics["recall_weighted"] = recall_score(
        y_true, y_pred, average="weighted", zero_division=0
    )

    # F1
    metrics["f1_macro"] = f1_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    metrics["f1_weighted"] = f1_score(
        y_true, y_pred, average="weighted", zero_division=0
    )

    # For ROC / PR AUC, we need to treat binary and multiclass separate so the array size matches correctly
    # Determine number of classes from labels
    num_unique = len(np.unique(y_true))

    try:
        if num_unique > 2:
            # --- MULTICLASS ---
            metrics["roc_auc_macro"] = roc_auc_score(
                y_true, y_pred_prob_mc, multi_class="ovr", average="macro"
            )
            metrics["roc_auc_weighted"] = roc_auc_score(
                y_true, y_pred_prob_mc, multi_class="ovr", average="weighted"
            )

            metrics["pr_auc_macro"] = average_precision_score(
                y_true, y_pred_prob_mc, average="macro"
            )
            metrics["pr_auc_weighted"] = average_precision_score(
                y_true, y_pred_prob_mc, average="weighted"
            )

        else:
            # --- BINARY ---
            prob_pos = y_pred_prob_mc[:, 1]  # positive-class probability/logit

            metrics["roc_auc_macro"] = roc_auc_score(y_true, prob_pos)
            metrics["roc_auc_weighted"] = metrics["roc_auc_macro"]

            metrics["pr_auc_macro"] = average_precision_score(y_true, prob_pos)
            metrics["pr_auc_weighted"] = metrics["pr_auc_macro"]

    except ValueError:
        metrics["roc_auc_macro"] = np.nan
        metrics["roc_auc_weighted"] = np.nan
        metrics["pr_auc_macro"] = np.nan
        metrics["pr_auc_weighted"] = np.nan


    return metrics

In [ ]:
# reload the best model
import os

model = monai.networks.nets.DenseNet121(spatial_dims=3, in_channels=1, out_channels=2).to(device)
model.load_state_dict(torch.load(MODEL_DIR / "best_metric_model_classification3d_array.pth"))
model.eval()

# recreate the test data loader
# should exist from the original build, but copy here just in case
def squeeze_to_one_channel(x):
    # if C==1, do nothing
    if x.shape[0] == 1:
        return x
    # if C>1 (e.g., DWI), keep channel 0
    return x[0:1, ...]  

val_transforms = Compose([ScaleIntensity(), 
                          EnsureChannelFirst(), 
                          Lambda(squeeze_to_one_channel),
                          Resize((96, 96, 96))])
test_ds = ImageDataset(image_files=image_files[label_groups['test']].tolist(), labels=image_labels[label_groups['test']].tolist(), transform=val_transforms)
test_loader = DataLoader(test_ds, batch_size=2, num_workers=8, pin_memory=pin_memory)
labels_true=image_labels[label_groups['test']].tolist()

In [ ]:
# loop over test data
import torch.nn.functional as F
import time
y_pred = []
y_true = []
t1 = time.time()
for test_data in test_loader:
    test_images, test_labels = test_data[0].to(device), test_data[1].to(device)
    with torch.no_grad():
        test_outputs = model(test_images)

        probs = F.softmax(test_outputs, dim=1)
        pred = torch.argmax(probs, dim=1)
        y_pred.extend(probs[:, 1].cpu().numpy())
        y_true.extend(test_labels.cpu().numpy())
t2 = time.time()
print(f'test time: {t2-t1}')


In [ ]:
metrics = compute_multiclass_metrics(np.asarray(y_true), np.asarray(y_pred))
print(metrics)